# MobileNetV2 vs MobileViT-XS on PlantVillage
Accuracy vs deployability of a lightweight CNN and a hybrid CNN-ViT for plant disease classification.

## Setup

In [ ]:
BRANCH = "feature/training"
DRIVE_DIR = "/content/drive/MyDrive/cnn-vit-research"
RUN_ABLATION = True
RUN_ROBUSTNESS = True
SMOKE = False  # tiny synthetic dataset, only for testing the pipeline

In [ ]:
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/cnn-vit-research-"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/SilverUmer21/cnn-vit-research-.git", REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40", "scikit-learn", "scipy"], check=True)
else:
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, REPO)

In [ ]:
import torch, transformers

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu")

## Data

In [ ]:
import glob, shutil, zipfile
from pathlib import Path

def find_data_root(base):
    for p in sorted(Path(base).rglob("train")):
        if p.is_dir() and (p.parent / "val").is_dir():
            return p.parent
    return None

if SMOKE:
    from src.experiments.smoke import make_smoke_data
    LOCAL = Path("/content/smoke" if IN_COLAB else "/tmp/plantvillage_smoke")
    make_smoke_data("splits/manifest.csv", LOCAL / "PlantVillage", LOCAL / "manifest.csv")
    DATA_ROOT, MANIFEST, PERSIST = LOCAL / "PlantVillage", LOCAL / "manifest.csv", LOCAL / "persist"
else:
    # reading 54k small files from Drive is slow, so unzip to the local disk once per session
    LOCAL = Path("/content/data")
    DATA_ROOT = find_data_root(LOCAL) if LOCAL.exists() else None
    if DATA_ROOT is None:
        zips = sorted(glob.glob(f"{DRIVE_DIR}/data/*.zip"))
        assert zips, f"no zip found in {DRIVE_DIR}/data/"
        LOCAL.mkdir(parents=True, exist_ok=True)
        shutil.copy(zips[0], "/content/plantvillage.zip")
        zipfile.ZipFile("/content/plantvillage.zip").extractall(LOCAL)
        os.remove("/content/plantvillage.zip")
        DATA_ROOT = find_data_root(LOCAL)
    MANIFEST, PERSIST = Path(REPO) / "splits/manifest.csv", Path(DRIVE_DIR)
print("data root:", DATA_ROOT)

In [ ]:
import pandas as pd
from src.experiments.config import (Paths, TrainConfig, BenchmarkConfig, SEEDS, MODELS, MODEL_ORDER,
                                    main_runs, ablation_runs, run_name)

paths = Paths(repo=REPO, persist=PERSIST, data_root=DATA_ROOT, manifest=MANIFEST).make()
manifest = pd.read_csv(paths.manifest)
missing = [p for p in manifest["path"] if not (paths.data_root / p).exists()]
assert not missing, f"{len(missing)} images missing, e.g. {missing[:3]}"
print(f"{len(manifest):,} images found")

CFG = TrainConfig()
if SMOKE:
    CFG = CFG.with_overrides(epochs=2, batch_size=16, eval_batch_size=32, lr=1e-3)
SEED_LIST = SEEDS if not SMOKE else (42,)
CFG

## 1. Data preparation
Fixed split from `splits/manifest.csv`: 80/10/10, stratified by class, with duplicate and near-duplicate
images kept in the same split.

In [ ]:
from src.experiments import plots as P, latex_tables as T, analysis as A

print(manifest.groupby("split").agg(images=("path", "size"), groups=("group_id", "nunique")).loc[["train", "val", "test"]])
print("groups in more than one split:", int((manifest.groupby("group_id")["split"].nunique() > 1).sum()))
A.write_tex(T.dataset_split(manifest), paths.tables / "split.tex")
P.class_distribution(manifest, paths.figures)
P.sample_grid(manifest, paths.data_root, paths.figures);

In [ ]:
# Member 1's loader checks, then our GPU preprocessing against Member 1's CPU recipe
if not SMOKE:
    r = subprocess.run([sys.executable, "-m", "src.dataset.check_loaders", str(paths.manifest), str(paths.data_root)],
                       capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-1500:])
sample = manifest.groupby("label").head(1)["path"].map(lambda p: str(paths.data_root / p)).tolist()
r = subprocess.run([sys.executable, "-m", "tests.test_preprocess", *sample], capture_output=True, text=True)
print(r.stdout, r.stderr[-1500:])
assert r.returncode == 0

## 2. Training
Same protocol for both models: AdamW (lr 1e-4, wd 1e-4), cosine schedule, batch 32, max 10 epochs,
early stopping on val loss (patience 5), best checkpoint by val loss, FP16 mixed precision.
Seeds 42, 43, 44, then the preprocessing ablation (seed 42).

In [ ]:
from src.experiments.train import train_run

def log(msg):
    print(msg, flush=True)

# finished runs are skipped, unfinished ones resume from the last epoch saved on Drive
todo = main_runs(SEED_LIST) + (ablation_runs() if RUN_ABLATION and not SMOKE else [])
for i, (m, s, v) in enumerate(todo, 1):
    log(f"\n=== {i}/{len(todo)}: {run_name(m, s, v)} ===")
    train_run(paths, m, s, v, CFG, device=DEVICE, log=log)
log("\ntraining done")

## 3. E1: Classification performance

In [ ]:
runs, preds = A.load_runs(paths)
main = A.main_results(runs)
per_seed = A.per_seed_table(runs)
stats = A.paired_statistics(preds)
cost = A.training_cost(runs)
for name, df in [("main", main), ("per_seed", per_seed), ("stats", stats), ("training_cost", cost)]:
    df.to_csv(paths.tables / f"{name}.csv", index=False)
A.write_tex(T.main_results(main), paths.tables / "main.tex")
A.write_tex(T.per_seed(per_seed, CFG.epochs, CFG.patience), paths.tables / "per_seed.tex")
A.write_tex(T.statistics(stats), paths.tables / "stats.tex")
A.write_tex(T.training_cost(cost), paths.tables / "training_cost.tex")
display(main.round(3)); display(per_seed.round(4)); display(stats.round(4)); display(cost.round(2))

In [ ]:
P.learning_curves(paths, runs, paths.figures);

In [ ]:
pc = A.per_class_table(preds)
pc.to_csv(paths.tables / "per_class.csv", index=False)
A.write_tex(T.per_class(pc), paths.tables / "per_class.tex")
P.per_class_f1(pc, paths.figures)
display(pc.sort_values("diff").round(4))

## 4. E2: Deployability
Both models benchmarked in the same session on the same hardware (seed-42 weights).

In [ ]:
from src.experiments import storage
from src.experiments.benchmark import benchmark_model
from src.experiments.qualitative import load_best

bench_file = paths.results / "benchmark.json"  # delete to re-measure
class_names = sorted(manifest["label"].unique())
if bench_file.exists():
    bench = storage.load_json(bench_file)
else:
    BCFG = BenchmarkConfig() if not SMOKE else BenchmarkConfig(runs=10, warmup=2, throughput_runs=3)
    bench = {}
    for m in MODEL_ORDER:
        model = load_best(paths, run_name(m, 42), m, class_names)
        bench[m] = benchmark_model(model, m, BCFG, log=log)
        del model
    storage.save_json(bench, bench_file)
dep = A.deployability(bench)
dep.to_csv(paths.tables / "deployability.csv", index=False)
A.write_tex(T.deployability(dep), paths.tables / "deployability.tex")
display(dep.round(3).T)

## 5. E3: Accuracy vs cost

In [ ]:
tr = A.tradeoff(main, dep)
tr.to_csv(paths.tables / "tradeoff.csv", index=False)
A.write_tex(T.tradeoff(tr), paths.tables / "tradeoff.tex")
display(tr.round(3))
P.tradeoff(main, dep, paths.figures);

## 6. E4: Qualitative results

In [ ]:
tc = A.top_confusions_all(preds)
tc.to_csv(paths.tables / "top_confusions.csv", index=False)
A.write_tex(T.top_confusions(tc), paths.tables / "top_confusions.tex")
display(tc)
for m in MODEL_ORDER:
    P.confusion_errors(A.mean_confusion(preds, m), class_names, m, paths.figures)

In [ ]:
from src.experiments import qualitative as Q

pair = Q.pair_for_seed(preds, 42)
P.example_grid(Q.select_examples(pair, "correct", 12), paths.data_root, paths.figures, "e4_correct",
               "Correctly classified test images")
for m in MODEL_ORDER:
    items = Q.select_examples(pair, m, 12)
    if items:
        P.example_grid(items, paths.data_root, paths.figures, f"e4_failures_{m}",
                       f"{MODELS[m]['display']}: misclassified test images, most confident first")
items = Q.select_examples(pair, "disagree", 12)
if items:
    P.example_grid(items, paths.data_root, paths.figures, "e4_disagreements",
                   "Test images only one of the two models gets right")

In [ ]:
cam_models = {m: load_best(paths, run_name(m, 42), m, class_names, DEVICE) for m in MODEL_ORDER}
rows = Q.gradcam_rows(cam_models, pair, Q.gradcam_indices(pair, 3, 3), paths.data_root, DEVICE)
P.gradcam_grid(rows, paths.figures);

## 7. E5: Robustness to image corruptions
All trained models on the test set with noise, blur, low brightness, low contrast, JPEG compression and
low resolution (3 severities each). No retraining.

In [ ]:
from src.experiments.robustness import evaluate_corruptions, summarise_robustness

rob_file = paths.tables / "robustness_raw.csv"
if RUN_ROBUSTNESS:
    if rob_file.exists():
        rob = pd.read_csv(rob_file)
    else:
        models = {r["run"]: (r["model"], load_best(paths, r["run"], r["model"], class_names, DEVICE))
                  for _, r in runs.iterrows()}
        rob = evaluate_corruptions(models, paths, DEVICE, num_workers=CFG.num_workers, log=log)
        storage.save_dataframe(rob, rob_file)
        del models
    rsum = summarise_robustness(rob)
    rsum.to_csv(paths.tables / "robustness.csv", index=False)
    clean = {m: A.mean_std(rob[(rob.corruption == "clean") & (rob.model == m)]["accuracy"]) for m in MODEL_ORDER}
    A.write_tex(T.robustness(rsum, clean), paths.tables / "robustness.tex")
    display(rsum.round(4))
    P.robustness(rob, paths.figures);

## 8. E6: Preprocessing ablation (ImageNet mean/std vs each model's own preprocessing)

In [ ]:
abl_runs, _ = A.load_runs(paths, variant="imagenet")
if len(abl_runs):
    ab = A.ablation_table(runs, abl_runs)
    ab.to_csv(paths.tables / "ablation.csv", index=False)
    A.write_tex(T.ablation(ab), paths.tables / "ablation.tex")
    display(ab.round(3))
else:
    print("no ablation runs yet")

## 9. Summary

In [ ]:
summary = {
    "main": main.to_dict("records"), "stats": stats.to_dict("records"), "training_cost": cost.to_dict("records"),
    "deployability": dep.to_dict("records"), "tradeoff": tr.to_dict("records"),
    "robustness": rsum.to_dict("records") if RUN_ROBUSTNESS else None,
    "ablation": ab.to_dict("records") if len(abl_runs) else None,
    "environment": storage.load_json(paths.run_dir(run_name(MODEL_ORDER[0], 42)) / "config.json")["environment"],
}
storage.save_json(summary, paths.results / "summary.json")
best = main.loc[main["macro_f1_mean"].idxmax()]
print(f"best macro-F1: {MODELS[best['model']]['display']} {best['macro_f1_mean']:.2f} ± {best['macro_f1_std']:.2f} %")

## Appendix: source code

In [ ]:
from IPython.display import Code

for f in ["config", "data", "models", "train", "evaluate", "benchmark", "robustness", "gradcam",
          "qualitative", "analysis", "latex_tables", "plots", "storage"]:
    display(Code(filename=f"src/experiments/{f}.py", language="python"))
for f in ["datasets", "preprocessing"]:
    display(Code(filename=f"src/dataset/{f}.py", language="python"))